In [5]:
from siriuspy.devices.beamline import Slit, DVFImg
import numpy as np
import time 
import h5py

# Constants

In [6]:
STEP = 0.1

# Devices initialization

In [8]:
slit1 = Slit(devname=Slit.DEVICES.SLIT1)

dvf1 = DVFImg(DVFImg.DEVICES.CAX_DVF1)
dvf2 = DVFImg(DVFImg.DEVICES.CAX_DVF2)

# Function definitions

In [4]:
def initialize_hdf5(fname, dv: dict):
    
    with h5py.File(fname, 'w') as f:
        f.attrs['begin time'] = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
        keys = list(dv.keys())
        for key in keys:
            f.attrs[key] = dv[key]

def append_dataset_hdf5(fname, img1_array, img2_array, tag, t0, positions):
    
    with h5py.File(fname, 'a') as f:
        dset = f.create_dataset('STEP_{0:03d}_1'.format(tag), data=img1_array, compression="gzip")
        dset.attrs['slit_top'] = positions[0]
        dset.attrs['slit_bottom'] = positions[1]
        dset.attrs['slit_right'] = positions[2]
        dset.attrs['slit_left'] = positions[3]
        dset.attrs['ellapsed time (s)'] = round(time.time() - t0, 3)

        dset2 = f.create_dataset('STEP_{0:03d}_2'.format(tag), data=img2_array, compression="gzip")
        dset2.attrs['slit_top'] = positions[0]
        dset2.attrs['slit_bottom'] = positions[1]
        dset2.attrs['slit_right'] = positions[2]
        dset2.attrs['slit_left'] = positions[3]
        dset2.attrs['ellapsed time (s)'] = round(time.time() - t0, 3)

# Single slit scan

### Key position functions

In [24]:
# Initial positions
reset_slits = lambda: (
    slit1.move_robust_top(value=18.21) or
    slit1.move_robust_bottom(value=35.3) or
    slit1.move_robust_left(value=47.8) or
    slit1.move_robust_right(value=43.1)
)
# Bordering positions
open_slits = lambda: (
    slit1.move_robust_top(value=19.71) or
    slit1.move_robust_bottom(value=35.8) or
    slit1.move_robust_left(value=47.2) or
    slit1.move_robust_right(value=43.6)
)

### HDF5 file initialization

In [ ]:
fname = 'single_slit_scan.h5'

dv = dict()

dv['slit_top'] = slit1.top_pos
dv['slit_bottom'] = slit1.bottom_pos
dv['slit_left'] = slit1.left_pos
dv['slit_right'] = slit1.right_pos

dv['STEP'] = 0.1

initialize_hdf5(fname, dv)

### Positions to sweep

In [25]:
slit_positions_top = np.arange(14.81, 19.71+STEP, STEP) #[-3:]
slit_positions_bottom = np.arange(30.9, 35.8+STEP, STEP) #[-3:]
slit_positions_right = np.arange(45.8, 47.2+STEP, STEP) #[-3:]
slit_positions_left = np.arange(42.3, 43.6+STEP, STEP) #[-3:]

slit_pos = [
    slit_positions_top,
    slit_positions_bottom,
    slit_positions_right,
    slit_positions_left,
]

slit_positions_top

array([14.81, 14.91, 15.01, 15.11, 15.21, 15.31, 15.41, 15.51, 15.61,
       15.71, 15.81, 15.91, 16.01, 16.11, 16.21, 16.31, 16.41, 16.51,
       16.61, 16.71, 16.81, 16.91, 17.01, 17.11, 17.21, 17.31, 17.41,
       17.51, 17.61, 17.71, 17.81, 17.91, 18.01, 18.11, 18.21, 18.31,
       18.41, 18.51, 18.61, 18.71, 18.81, 18.91, 19.01, 19.11, 19.21,
       19.31, 19.41, 19.51, 19.61, 19.71, 19.81])

### Scan

In [ ]:
open_slits()

tag = 0
for side in ('top', 'bottom', 'left', 'right'):
    positions = locals()['slit_positions_{}'.format(side)]
    
    for pos in positions:
        t0 = time.time()

        method_name = 'move_robust_{}'.format(side)
        move_slit = getattr(slit1, method_name)
        move_slit(value=pos)

        pos_current = [
            slit1.top_pos,
            slit1.bottom_pos,
            slit1.left_pos,
            slit1.right_pos,
        ]
        img1 = dvf1.image
        img2 = dvf2.image

        append_dataset_hdf5(fname, img1, img2, tag, t0, pos_current)

        tag += 1

    open_slits()

    print('Finished {:} slit scan'.format(side))

reset_slits()

# Square Scan

In [27]:
GAP_HOR = 0.4
GAP_VER = 0.4

# positions to set the square
slit_top_pos0 = 14.81 + GAP_VER
slit_bottom_pos0 = 35.8
slit_right_pos0 = 45.8 + GAP_HOR
slit_left_pos0 = 43.6

### Key position function

In [ ]:
# Initial positions
set_initial_square = lambda: (
    slit1.move_robust_top(slit_top_pos0) or
    slit1.move_robust_bottom(slit_bottom_pos0) or
    slit1.move_robust_left(slit_left_pos0) or
    slit1.move_robust_right(slit_right_pos0)
)

### HDF5 file initialization

In [ ]:
fname = 'square_scan.h5'

dv = dict()

dv['slit_top'] = slit1.top_pos
dv['slit_bottom'] = slit1.bottom_pos
dv['slit_left'] = slit1.left_pos
dv['slit_right'] = slit1.right_pos

initialize_hdf5(fname, dv)

### Positions to sweep

In [ ]:
xdeltas = np.linspace(0, 1.3, 6)
ydeltas = np.linspace(0, 4.9, 12)

### Scan

In [ ]:
open_slits()

tag = 0
for j, ydelta in enumerate(ydeltas):
    slit1.move_robust_top(value=slit_top_pos0+ydelta)
    slit1.move_robust_bottom(value=slit_bottom_pos0-ydelta)
    slit1.move_robust_left(value=slit_left_pos0)
    slit1.move_robust_right(value=slit_right_pos0)

    for i, xdelta in enumerate(xdeltas):
        slit1.move_robust_left(value=slit_left_pos0-xdelta)
        slit1.move_robust_right(value=slit_right_pos0+xdelta)

        pos_current = [
            slit1.top_pos,
            slit1.bottom_pos,
            slit1.left_pos,
            slit1.right_pos,
        ]
        img1 = dvf1.image
        img2 = dvf2.image

        append_dataset_hdf5(fname, img1, img2, tag, t0, pos_current)
        tag += 1

reset_slits()